# Lab 8 Homework: Model Selection
### Cross-Validation and Hyper-parameter Tuning

This homework **continues the Lab 8 classwork**. You will use the same tools on new data and explain what you see.

| Question | Topic | Dataset | Marks |
|---|---|---|---|
| Q1 | Is my score just luck? One split vs k-fold vs LOOCV | Moons | 2 |
| Q2 | Choosing `n_neighbors` with `GridSearchCV` | Moons | 2 |
| Q3 | Choosing the right CV iterator | Wine, Activity sensors | 2 |
| Q4 | Comparing models with `cross_validate` | Wine | 2 |
| Q5 | Grid vs Random vs Successive Halving | Wine | 2 |

**Instructions**
- Some code cells are **partly written**. Replace every `___` with your code. Other cells are empty: write the code yourself (the classwork has very similar examples).
- Answer the questions under **Your answer:** in 2–4 short sentences. **Use numbers from your own output.**
- Keep the given `random_state` values.
- Before you submit: **Kernel → Restart & Run All**, check there are no errors, and save.
- Submit the file as `Lab8HW_<StudentID>_<Name>.ipynb`.

## Setup (run this first)

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    KFold, StratifiedKFold, GroupKFold, LeaveOneOut,
    GridSearchCV, RandomizedSearchCV,
)
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)

---
## Data for Q1 and Q2: Moons

`make_moons` creates two classes shaped like two half-moons, with some noise. It has only **2 features**, so we can plot it.
We keep 25 % as a **test set** and do not touch it until the end of Q2.

In [ ]:
X_moon, y_moon = datasets.make_moons(n_samples=500, noise=0.35, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_moon, y_moon, test_size=0.25,
                                                    stratify=y_moon, random_state=42)
print("train:", X_train.shape, " test:", X_test.shape)

---
## Q1. Is my score just luck? (2 marks)

In the classwork (Parts 2–4) you saw that one validation split can give a lucky or unlucky score. Here you will **measure** this.

We use **one fixed model**: `KNeighborsClassifier(n_neighbors=15)`, and only the **training data** (`X_train`, `y_train`).

### Q1(a) — Run four methods (1.25 marks)
Complete the code. Replace every `___`.
1. **One validation split**, repeated with 10 seeds (`random_state = 0 … 9`), `test_size=0.2`.
2. **5-fold CV**, repeated with 10 seeds: `KFold(n_splits=5, shuffle=True, random_state=seed)`.
3. **10-fold CV**, repeated with 10 seeds: `KFold(n_splits=10, shuffle=True, random_state=seed)`.
4. **LOOCV**, only once (it has no randomness).

In [ ]:
model = KNeighborsClassifier(n_neighbors=15)

one_split, fold5, fold10 = [], [], []
for seed in range(10):
    # 1) one validation split
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)
    model.fit(___, ___)
    one_split.append(model.score(___, ___))

    # 2) 5-fold CV  (save the MEAN of the 5 fold scores)
    scores = cross_val_score(model, X_train, y_train, cv=___)
    fold5.append(scores.mean())

    # 3) 10-fold CV (save the MEAN of the 10 fold scores)
    scores = ___
    fold10.append(scores.mean())

# 4) LOOCV (once)
start = time.time()
loocv = ___
print(f"LOOCV accuracy: {loocv:.3f}  (time: {time.time() - start:.2f} s)")

results_q1 = pd.DataFrame({"One split": one_split, "5-fold CV": fold5, "10-fold CV": fold10})
results_q1.agg(["mean", "std", "min", "max"])

### Q1(b) — Box plot (0.25 mark)
Draw a **box plot** of `results_q1` (one box per method) and add the LOOCV accuracy as a red dashed horizontal line.
Hint: `results_q1.boxplot()` and `plt.axhline(loocv, color="red", linestyle="--", label="LOOCV")`.

In [ ]:
# Code Here

### Q1(c) — Questions (0.5 mark)
1. Which method gives the **most different** scores from seed to seed? Which gives the most **stable** scores? (use the `std` row)
2. Why is there no box for LOOCV?
3. In one sentence: why do we use cross-validation instead of one validation split?

**Your answer:**

---
## Q2. Choosing `n_neighbors` with `GridSearchCV` (2 marks)

### Q2(a) — Look at the data (0.25 mark)
Make a **scatter plot** of `X_train` (feature 0 on the x-axis, feature 1 on the y-axis), coloured by `y_train`.
Hint: `plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", alpha=0.6)`.

In [ ]:
# Code Here

### Q2(b) — Grid search (0.75 mark)
Complete the code. Search `n_neighbors` = 1, 5, 9, …, 97 with **10-fold** stratified CV and keep the train scores.
Then print the best parameter, the best CV score and the **test-set** score.

In [ ]:
param_grid = {"n_neighbors": np.arange(1, 100, 4)}
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid = GridSearchCV(KNeighborsClassifier(), param_grid=___, cv=___, return_train_score=True)
grid.fit(___, ___)

print("best parameters :", grid.best_params_)
print(f"best CV score   : {___:.3f}")
print(f"test-set score  : {grid.score(___, ___):.3f}")

### Q2(c) — Line plot (0.25 mark)
From `grid.cv_results_`, plot `mean_train_score` and `mean_test_score` against `n_neighbors` on the same figure (two lines, with a legend).
Hint: `results = pd.DataFrame(grid.cv_results_)`, then `plt.plot(results["param_n_neighbors"].astype(int), results["mean_train_score"], label="train")`.

In [ ]:
# Code Here

### Q2(d) — Questions (0.75 mark)
1. At `n_neighbors = 1`, what are the train and validation scores? Is the model **overfitting** or **underfitting**?
2. What happens to both scores when `n_neighbors` becomes very large (e.g. 97)?
3. Why do we use the test set **only once**, at the very end?

**Your answer:**

---
## Q3. Choosing the right CV iterator (2 marks)

### Q3(a) — Sorted data (1 mark)
The **Wine** dataset (178 wines, 3 classes, 13 features) is **sorted by class**, like Iris in the classwork.

In [ ]:
X_wine, y_wine = datasets.load_wine(return_X_y=True)
print("first 10 labels:", y_wine[:10])
print("last 10 labels :", y_wine[-10:])

knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

Complete the code: compute the mean CV accuracy of `knn_scaled` on **all** the wine data (`X_wine`, `y_wine`) with the three CV methods below, then draw a **bar chart** of the three means.

In [ ]:
cv_methods = {
    "KFold(3)": KFold(n_splits=3),
    "KFold(3, shuffle)": KFold(n_splits=3, shuffle=True, random_state=42),
    "StratifiedKFold(3, shuffle)": StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
}
mean_scores = {}
for name, cv in cv_methods.items():
    scores = ___
    print(f"{name:28s} fold scores: {np.round(scores, 3)}")
    mean_scores[name] = scores.mean()

# Bar chart
plt.bar(___, ___)
plt.ylabel("mean CV accuracy"); plt.xticks(rotation=15); plt.show()

**Question:** Why is `KFold(3)` so bad here? Which method would you use?

**Your answer:**

### Q3(b) — Grouped data (1 mark)
A fitness app classifies **sitting (0) vs walking (1)** from 6 sensor features. We have **30 people**, 20 recordings each. The company wants to know the accuracy for **new customers** (people the model has never seen).

In [ ]:
def make_activity_data(n_people=30, recordings_per_activity=10, seed=0):
    # Do not change. Each person moves in their own way, so recordings of the same person look alike.
    rng = np.random.RandomState(seed)
    X, y, person = [], [], []
    for p in range(n_people):
        style = rng.normal(0, 2, 6)                  # how this person moves in general
        walk_effect = 2.0 + rng.normal(0, 1.5, 6)    # how walking differs from sitting for this person
        for activity in [0, 1]:
            for _ in range(recordings_per_activity):
                X.append(style + activity * walk_effect + rng.normal(0, 0.4, 6))
                y.append(activity); person.append(p)
    return np.array(X), np.array(y), np.array(person)

X_act, y_act, person = make_activity_data()
print(X_act.shape, "| people:", len(np.unique(person)))
rf = RandomForestClassifier(n_estimators=100, random_state=42)

Compute the mean accuracy of `rf` with
- `KFold(n_splits=5, shuffle=True, random_state=42)`, and
- `GroupKFold(n_splits=5)` (remember `groups=person`),

then draw a **bar chart** of the two means.

In [ ]:
# Code Here

**Question:** Which accuracy should the company report for new customers? Why is the other one too high?

**Your answer:**

---
## Q4. Comparing models with `cross_validate` (2 marks)

Which algorithm works best for the **Wine** data? Note: the features have very different scales (e.g. *proline* ≈ 750, *hue* ≈ 1).

### Q4(a) — Compare five models (1.25 marks)
Complete the code. For every model, run `cross_validate` with `cv=cv5`, `scoring=["accuracy", "f1_macro"]` and `return_train_score=True`, and save the mean results.
Then show the table and draw a **bar chart** of train vs test accuracy.

In [ ]:
models = {
    "KNN (no scaling)": KNeighborsClassifier(),
    "KNN (scaled)": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, m in models.items():
    res = cross_validate(___)
    rows.append({"model": name,
                 "train_acc": res["train_accuracy"].mean(),
                 "test_acc": res[___].mean(),
                 "test_f1_macro": res[___].mean(),
                 "fit_time": res["fit_time"].mean()})

summary_q4 = pd.DataFrame(rows).set_index("model")
display(summary_q4)

# Bar chart: train vs test accuracy for each model
summary_q4[["train_acc", "test_acc"]].plot.bar(figsize=(8, 4), rot=15)
plt.ylabel("accuracy"); plt.ylim(0.5, 1.05); plt.show()

### Q4(b) — Questions (0.75 mark)
1. Which model would you choose? Use the numbers.
2. Why is KNN much better after scaling?
3. Which model has train accuracy 1.00 but a much lower test accuracy? What does that mean?

**Your answer:**

---
## Q5. Grid vs Random vs Successive Halving (2 marks)

We tune a Random Forest on the Wine data. First we keep a test set.

In [ ]:
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_wine, y_wine, test_size=0.25,
                                                        stratify=y_wine, random_state=42)
rf_grid = {"n_estimators": [50, 100],
           "max_depth": [2, 4, None],
           "min_samples_leaf": [1, 5]}
print(Xw_train.shape, Xw_test.shape)

### Q5(a) — Count the fits by hand (0.25 mark)
With `cv=5`: how many fits does `GridSearchCV` make with `rf_grid`? And `RandomizedSearchCV` with `n_iter=5`? (Do not count the final refit.)

**Your answer:**

### Q5(b) — Run the three searches (1 mark)
Complete the code, then draw a **bar chart** of the time used by each method.

In [ ]:
searches = {
    "GridSearchCV": GridSearchCV(RandomForestClassifier(random_state=42), rf_grid, cv=5),
    "RandomizedSearchCV": RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_grid,
                                             n_iter=___, cv=5, random_state=42),
    "HalvingGridSearchCV": HalvingGridSearchCV(RandomForestClassifier(random_state=42), rf_grid,
                                               cv=5, factor=3, random_state=42),
}

rows = []
for name, search in searches.items():
    start = time.time()
    search.fit(___, ___)
    rows.append({"method": name,
                 "time_s": time.time() - start,
                 "n_fits": len(search.cv_results_["params"]) * 5,
                 "best_params": search.best_params_,
                 "best_cv_score": ___,
                 "test_score": search.score(___, ___)})

summary_q5 = pd.DataFrame(rows).set_index("method")
display(summary_q5)

halving = searches["HalvingGridSearchCV"]
print("Halving - candidates in each round:", halving.n_candidates_)
print("Halving - samples in each round   :", halving.n_resources_)

# Bar chart of the time
# Code Here

### Q5(c) — Questions (0.75 mark)
1. Do the `n_fits` in your table match your answer in Q5(a)?
2. Explain the halving rounds in your own words, using `n_candidates_` and `n_resources_`.
3. If the dataset had **1 million rows** and 500 candidate settings, which method would you choose and why?

**Your answer:**

---
### ✅ Before you submit
- [ ] Every `___` is replaced and **Kernel → Restart & Run All** works without errors
- [ ] Every **Your answer:** is filled in with numbers from your output
- [ ] File name: `Lab8HW_<StudentID>_<Name>.ipynb`